import pandas as pd

import numpy as np

import statsmodels.api as sm

import itertools

import math

import sympy as sp

\# ==========================================

\# PROBLEM 3: MODEL SELECTION

\# ==========================================

print("==========================================")

print("PROBLEM 3: MODEL SELECTION")

print("==========================================\n")

\# 1. Load the Dataset

\# Ensure 'FE-GWP1_model_selection_2.csv' is in the same directory as this script.

try:

df = pd.read_csv('FE-GWP1_model_selection_2.csv')

\# Assuming columns are named exactly 'Y', 'Z1', 'Z2', 'Z3', 'Z4', 'Z5'

y = df\['Y'\]

X_full = df\[\['Z1', 'Z2', 'Z3', 'Z4', 'Z5'\]\]

print("Dataset loaded successfully.\n")

except FileNotFoundError:

print("Error: 'FE-GWP1_model_selection_2.csv' not found.")

print("Please ensure the CSV file is in the same directory.")

\# Exiting the script if data isn't found to prevent downstream errors

exit()

\# ------------------------------------------

\# Approach 1: Backward Elimination

\# ------------------------------------------

print("--- Approach 1: Backward Elimination ---")

def backward_elimination(y, X, significance_level=0.05):

features = X.columns.tolist()

while len(features) \> 0:

X_with_const = sm.add_constant(X\[features\])

model = sm.OLS(y, X_with_const).fit()

p_values = model.pvalues\[1:\] \# Exclude intercept

max_p_value = p_values.max()

if max_p_value \> significance_level:

excluded_feature = p_values.idxmax()

print(f"Removing '{excluded_feature}' (p-value: {max_p_value:.4f})")

features.remove(excluded_feature)

else:

break

print("\nFinal Reduced Model (Backward Elimination):")

final_model = sm.OLS(y, sm.add_constant(X\[features\])).fit()

print(final_model.summary())

return final_model

final_backward_model = backward_elimination(y, X_full)

\# ------------------------------------------

\# Approach 2: Forward Selection (Best Subsets based on AIC/BIC)

\# ------------------------------------------

print("\n--- Approach 2: Best Subsets (AIC/BIC Tracking) ---")

def get_best_models_by_size(y, X):

features = X.columns.tolist()

\# Null Model

null_model = sm.OLS(y, sm.add_constant(pd.Series(np.zeros(len(y)), name='Null'))).fit()

print(f"Null Model (Y ~ 1): AIC = {null_model.aic:.1f}, BIC = {null_model.bic:.1f}")

\# Iterate through sizes 1 to 5

for k in range(1, len(features) + 1):

best_aic = np.inf

best_bic = np.inf

best_combo = None

\# Check every combination of size k

for combo in itertools.combinations(features, k):

X_subset = sm.add_constant(X\[list(combo)\])

model = sm.OLS(y, X_subset).fit()

if model.aic \< best_aic:

best_aic = model.aic

best_combo = combo

if model.bic \< best_bic:

best_bic = model.bic

print(f"Best {k}-variable model (Y ~ {' + '.join(best_combo)}): AIC = {best_aic:.1f}, BIC = {best_bic:.1f}")

get_best_models_by_size(y, X_full)

\# ==========================================

\# PROBLEM 4: REGRESSION INTERPRETATION

\# ==========================================

print("\n==========================================")

print("PROBLEM 4: MATH CALCULATIONS")

print("==========================================\n")

\# ------------------------------------------

\# Question 4f: At what age is salary highest?

\# ------------------------------------------

print("--- Question 4f: Peak Salary Age ---")

\# Using SymPy to calculate the partial derivative and set to 0

age = sp.Symbol('age')

\# Equation: ln(wage) = ... + 0.19\*age - 0.002\*age^2 + ...

\# Derivative with respect to age: d(ln(wage))/d(age) = 0.19 + 2\*(-0.002)\*age

beta_age = 0.19

beta_age2 = -0.002

derivative_eq = beta_age + 2 \* beta_age2 \* age

\# Solve for age where derivative equals 0

optimal_age = sp.solve(derivative_eq, age)\[0\]

print(f"Derivative Equation: {derivative_eq} = 0")

print(f"Calculated Peak Age: {optimal_age} years\n")

\# ------------------------------------------

\# Question 4g: Average impact of advanced degree

\# ------------------------------------------

print("--- Question 4g: Impact of Advanced Degree ---")

\# Formula for percentage change of a dummy variable in a log-level model: (e^beta - 1) \* 100

beta_degree = 0.22

exact_impact = (math.exp(beta_degree) - 1) \* 100

linear_approx = beta_degree \* 100

print(f"Beta coefficient for Degree: {beta_degree}")

print(f"Exact Percentage Change: (e^{beta_degree} - 1) \* 100 = {exact_impact:.2f}%")

print(f"Simple Linear Approximation: {linear_approx:.2f}%\n")